# A1_repeat — full 250 epochs, private Drive results

Choose **A100**, then **Run all** and complete ordinary Google Drive authorization. Arm, folds and paths are built in. Rerun the same file after a disconnect: verified checkpoints resume automatically; verified completed fits are skipped.

The first real epochs are the smoke check. There are no CU balance prompts, reservations, two-hour segments or deliberate stops. Reported Sprint12 training time is 2.69–2.74 hours per fold before overhead; at the reported 6.77 CU/hour this is an estimate of 18.2–18.5 CU per fold before overhead, not a guarantee.

Run fold files in order0–4: start two sessions, then fill further available slots, up to five distinct folds. Do not run the all-fold file alongside them. 

Development99 only; excluded49 payloads stay untouched. No4D CTP. Exploratory development results; one repeat does not estimate training variance. Results stay private.

Input reference is bound; missing or changed Drive metadata fails closed. Code review does not establish patient results.


## Settings

In [ ]:
# Built-in delivery constants are in cell4; setup is automatic in the driver.


### Pipeline file: setup  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
DELIVERY_ARM = "A1_repeat"
DELIVERY_FOLDS = (4,)

ZSCORE_MIN_SD = 0.01
HISTEQ_BINS = 256


### Pipeline file: sprint 8 utilities and scoring (verbatim)  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
%%writefile /content/sprint13_pipeline.py
K_DJ='dataset.json'
K_AR='A1_repeat'
K_AZ='A1_zscore'
K_F='fallback'
K_AH='A1_histeq'
K_3F='3d_fullres'
K_MM='memory_mib'
K_P='preprocessing'
K_E='experiment'
K_ACMC='A1_ct_maps_cta'
K_SFJ='splits_final.json'
K_MJ='metrics.json'
K_NE='next_epoch'
K_AM='A1_multiwin'
K_R='resources'
K_AP='A1_pnormct'
K_EP='execution_plan'
K_V='validation'
K_LF='lesion_full'
K_MS='module_sha256'
K_BH='baseline_handoff'
K_BPJ='baseline/plans.json'
K_BHJ='baseline/handoff.json'
K_NS='normalization_schemes'
K_C='configurations'
K_N='NoNormalization'
K_ST='synthetic.txt'
K_DPPV='declared-preprocessing-partitions/v1'
K_EPS='execution_plan_sha256'
K_PCS='preprocessing_code_sha256'
K_DFJ='dataset_fingerprint.json'
K_LNB='loader_not_bottleneck'
K_PN='plans_name'
K_ICS='input_contract_sha256'
K_SS='split_sha256'
K_PI='preprocessing_id'
K_ES='environment_sha256'
K_SCS='source_capture_sha256'
K_LW='loader_wait'
K_PS='plans_sha256'
K_TS='timeout_seconds'
K_CS='cohort_sha256'
K_ERS='environment_requirements_sha256'
K_TI='training_iterations'
K_VI='validation_iterations'
K_ER='environment_requirements'
K_VC='validation_checks'
K_FIPPC='foreground_intensity_properties_per_channel'
from pathlib import Path
import hashlib,json,math,os,time,unittest,tempfile,copy,importlib
BASE_PLANS_SHA256='532aa8160bf0db6c83ceeb0c1456d3bf6f2f7b528cdb0a9586866ac724de92d0'
BASE_HANDOFF_SHA256='5a5649e12a568ce99c8c2b4abd88732d07a862d44c32da9f1ddfc599699ddb01'
RUN_ID='colab-1ad5fafc220b3d65f576ce08'
PARTITIONS_SHA256='2c197a9addf40af4d75dd220a095b62130b0ca58ffbe5dce8eb73dff570f64f8'
COHORT_SHA256='45c5746f60e2275383a2c4f4096295fe7a171da85a73737c5a35a8cc6fc9fe86'
SPLIT_SHA256='d339296ca3b1672fc744e14188b36c1860b379a38b807eca67d8b73038007c86'
ARMS_ORDER=(K_AR,'A1_repeat2',K_AZ,K_AH,K_AM,'A1_L',K_AP,'A1_pnorm_v2')
CANDIDATES=(K_AZ,K_AH,K_AM,'A1_L',K_AP)
WINDOWS={'ncct':(0,80),'cbf':(0,35),'cbv':(0,10),'mtt':(0,20),'tmax':(0,7),'cta':(0,90),'ncct_w2':(20,50),'cbf_w2':(0,100),'cbv_w2':(0,4),'mtt_w2':(0,12),'tmax_w2':(0,20),'cta_w2':(0,300)}
BASE6=('ncct','cbf','cbv','mtt','tmax','cta')
PERF_KINDS=('cbf','cbv','mtt','tmax')
ZSCORE_MIN_SD=0.01
HISTEQ_BINS=256
VOXEL_MM=(2.0,2.0,2.0)
PNORM_NN={'min_ref_voxels':20000,'min_scale':{'cbf':0.7,'cbv':0.2,'mtt':0.4,'tmax':0.14,'ncct':1.6},'clip_z':5.0}
FEATURE_CFG={'target_mm':2.0,'vessel_pct':98.0,'core_thr':0.3,'penumbra_tmax':6.0,'schema':'isles24-features-2mm-v2'}
import glob,traceback,uuid
CACHE_SCHEMA='isles24-features-2mm-v2'
FEATURE_NAMES=['cbf','cbv','mtt','tmax','ncct_hu','rel_cbf','rel_tmax','rel_hu','tissue_class','z_cbf','z_cbv','z_tmax','z_hu','dist_occlusion_mm','affected_side','core_rule','penumbra_rule','local_mean_tmax','local_std_tmax','local_mean_relcbf','slice_pos']
import os,json,time,glob,uuid,datetime
COLAB_RUN_ID='colab-1ad5fafc220b3d65f576ce08'
BASE_INPUT_SHA256='b84b5f872711720d16b6d7f8858b3bff65a0543ff0580195eaf228f20878c0b2'
import types
TRAINER_SOURCE_SHA256="7096efb2040135eb60df3c8bf39cdbcc373e299671fcee365b3aff2f4da4dfbd"

def config_hash(cfg: dict) -> str:
    """Short stable hash of the settings that change what a cache file contains."""
    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:12]
def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    s = a.sum() + b.sum()
    return 1.0 if s == 0 else float(2.0 * (a & b).sum() / s)
def load_cache(path, expect_hash, expect_case=None):
    """Load and VALIDATE a cache: schema, config hash, case, shapes, finite values. Raises on any mismatch."""
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    if meta.get("schema") != CACHE_SCHEMA or meta.get("config_hash") != expect_hash or meta.get("features") != FEATURE_NAMES:
        raise ValueError(f"cache {os.path.basename(path)} does not match schema/config ({meta.get('schema')}, {meta.get('config_hash')})")
    if expect_case and meta.get("case") != expect_case:
        raise ValueError(f"cache {os.path.basename(path)} belongs to {meta.get('case')}, expected {expect_case}")
    shape = tuple(int(s) for s in z["mask_shape"]); n = int(np.prod(shape))
    mask = np.unpackbits(z["mask"])[:n].reshape(shape).astype(bool)
    lesion_full = np.unpackbits(z["lesion_full"])[:n].reshape(shape).astype(bool)
    X = z["X"].astype(np.float32)
    if X.shape != (int(mask.sum()), len(FEATURE_NAMES)) or not np.isfinite(X).all():
        raise ValueError(f"cache {os.path.basename(path)} has bad shape or non-finite values")
    return {"X": X, "y": z["y"], "sdt": z["sdt"].astype(np.float32), "mask": mask, "lesion_full": lesion_full,
            "shape": shape, "healthy_side": z["healthy_side"], "affine": z["affine"],
            "voxel_ml": float(z["voxel_ml"]), "summary": json.loads(str(z["summary"])), "meta": meta}
def official_lesion_metrics(truth, pred, empty_value=1.0):
    """Exact semantics of the pinned official `compute_dice_f1_instance_difference`: 26-connected components (cc3d default) on truth and
    prediction; overlapping (truth, pred) component pairs scored by IoU, sorted descending, matched greedily one-to-one when IoU >= 0.2;
    F1 = TP / (TP + 0.5 FP + 0.5 FN); lesion-count difference = |n_truth - n_pred|; Dice = global binary Dice; both-empty -> empty_value."""
    from scipy.ndimage import label as _lab
    truth = np.asarray(truth).astype(bool); pred = np.asarray(pred).astype(bool); s = np.ones((3, 3, 3), bool)
    lt, nt = _lab(truth, structure=s) if truth.any() else (np.zeros(truth.shape, np.int32), 0); lp, npd = _lab(pred, structure=s) if pred.any() else (np.zeros(pred.shape, np.int32), 0)
    count_diff = abs(nt - npd)
    if nt == 0 and npd == 0: return float(empty_value), int(count_diff), float(empty_value)
    inter = float((truth & pred).sum()); dsc = 2 * inter / (truth.sum() + pred.sum()) if (truth.sum() + pred.sum()) > 0 else float(empty_value)
    pairs = []
    if nt and npd:
        both = truth & pred; rl, pl = lt[both], lp[both]; combos = np.unique(np.stack([rl, pl], 1), axis=0) if both.any() else np.zeros((0, 2), int)
        vt = np.bincount(lt.ravel()); vp = np.bincount(lp.ravel())
        for r, p in combos:
            i = float(((lt == r) & (lp == p)).sum()); u = float(vt[r] + vp[p] - i); pairs.append((i / u if u > 0 else 0.0, int(r), int(p)))
    pairs.sort(key=lambda x: -x[0]); mr, mp, tp = set(), set(), 0
    for iou, r, p in pairs:
        if r in mr or p in mp: continue
        if iou >= 0.2: mr.add(r); mp.add(p); tp += 1
    fp, fn = npd - tp, nt - tp; f1 = 0.0 if tp == 0 else tp / (tp + 0.5 * fp + 0.5 * fn)
    return float(f1), int(count_diff), float(dsc)
def asd_mm(pred, truth, spacing):
    # pooled bidirectional nearest-boundary distances; undefined when either mask is empty
    if pred.sum() == 0 or truth.sum() == 0: return np.nan
    def surface(m): return m & ~binary_erosion(m)
    sp, st = surface(pred), surface(truth); dt_t = distance_transform_edt(~st, sampling=spacing); dt_p = distance_transform_edt(~sp, sampling=spacing)
    return float(np.concatenate([dt_t[sp], dt_p[st]]).mean())
def patient_scores(c, p, pred):
    truth = c["lesion_full"]; mask = c["mask"]; ml = c["voxel_ml"]
    tp = int((pred & truth).sum()); fp = int((pred & ~truth).sum()); fn = int((~pred & truth).sum()); tn = int((~pred & ~truth & mask).sum())
    dsc = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    tv, pv = float(truth.sum() * ml), float(pred.sum() * ml); lf1, cdiff, _ = official_lesion_metrics(truth, pred, empty_value=1.0)
    y = truth[mask].astype(int); mask_pos, mask_neg = int(y.sum()), int(y.size - y.sum()); auc = float(roc_auc_score(y, p)) if 0 < mask_pos < y.size else np.nan; ap = float(average_precision_score(y, p)) if 0 < mask_pos < y.size else np.nan   # undefined when the mask holds no lesion voxel (or no non-lesion voxel)
    cov = float((truth & mask).sum() / truth.sum()) if truth.sum() else np.nan; bound = 2 * cov / (1 + cov) if truth.sum() else np.nan
    return {"dice": dsc, "f1_voxel": dsc, "precision": tp / (tp + fp) if (tp + fp) else np.nan, "recall": tp / (tp + fn) if (tp + fn) else np.nan, "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "lesion_ml": tv, "pred_ml": pv, "abs_vol_err_ml": abs(pv - tv), "signed_vol_err_ml": pv - tv, "lesion_ge5ml": tv >= 5.0, "pred_empty": bool(pred.sum() == 0), "truth_empty": bool(truth.sum() == 0),
            "lesion_f1": lf1, "lesion_count_diff": cdiff, "asd_mm": asd_mm(pred, truth, VOXEL_MM), "fm_auroc": auc, "fm_avg_precision": ap,
            "lesion_coverage_by_mask": cov, "lesion_ml_outside_mask": float((truth & ~mask).sum() * ml), "mask_dice_bound": bound, "mask_pos": mask_pos, "mask_neg": mask_neg, "tp": tp, "fp": fp, "fn": fn}
def _fx():
    z = lambda: np.zeros((6, 6, 3), bool); out = {}
    t = z(); p = z(); out["both_empty"] = official_lesion_metrics(t, p)[:2]
    t = z(); t[1:3, 1:3, 1] = 1; p = z(); out["truth_only"] = official_lesion_metrics(t, p)[:2]
    t = z(); t[0, 0, 0] = 1; t[1, 1, 1] = 1; p = z(); p[0, 0, 0] = 1; p[1, 1, 1] = 1; out["diagonal_contact_one_piece"] = official_lesion_metrics(t, p)[:2]
    t = z(); t[0:4, 0:4, 1] = 1; p = z(); p[0:2, 0:4, 1] = 1; p[3:4, 0:4, 1] = 1; out["truth_one_pred_two_split"] = official_lesion_metrics(t, p)[:2]
    t = z(); t[0:2, 0:6, 1] = 1; p = z(); p[0:2, 0:6, 1] = 1; p[3:5, 0:6, 1] = 1; out["one_matched_one_fp"] = official_lesion_metrics(t, p)[:2]
    t = z(); t[0:5, 0, 0] = 1; p = z(); p[0:1, 0, 0] = 1; out["iou_0.2_boundary"] = official_lesion_metrics(t, p)[:2]
    return out


### Pipeline file: library and run identity  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
%%writefile -a /content/sprint13_pipeline.py
def coverage_support(values,brain,coverage=None):
 values=np.asarray(values)
 brain=np.asarray(brain,dtype=bool)
 if values.shape!=brain.shape:
  raise ValueError('brain shape mismatch')
 support=brain&np.isfinite(values)
 if coverage is not None:
  coverage=np.asarray(coverage)
  if coverage.shape!=values.shape or not np.isfinite(coverage).all()or(not np.isin(coverage,[0,1]).all()):
   raise ValueError('invalid spatial coverage mask')
  support&=coverage.astype(bool)
 return(support,{'coverage_known':coverage is not None,'brain_n':int(brain.sum()),'support_n':int(support.sum()),'nonfinite_in_brain':int((brain&~np.isfinite(values)).sum())})
def zscore_in_brain(x,brain):
 x=np.asarray(x,dtype=float)
 support,_=coverage_support(x,brain)
 v=x[support]
 out=np.zeros_like(x,dtype=np.float32)
 m,s=(float(v.mean()),float(v.std()))if v.size else(0.0,0.0)
 su=max(s,ZSCORE_MIN_SD)
 rec={'mean':m,'sd':s,'sd_used':su,'n':int(v.size),'fallback':None}
 if not v.size or s<1e-06:
  rec['fallback']='empty support'if not v.size else'constant support'
  return(out,rec)
 out[support]=(v-m)/su
 rec.update({'fallback':'SD floored'if s<ZSCORE_MIN_SD else None,'z_min':float(out[support].min()),'z_max':float(out[support].max())})
 return(out,rec)
def histeq_in_brain(x,brain,nbins=None):
 x=np.asarray(x,dtype=float)
 support,_=coverage_support(x,brain)
 v=x[support]
 out=np.zeros_like(x,dtype=np.float32)
 if not v.size or float(v.std())<1e-06:
  return(out,{'fallback':'empty support'if not v.size else'constant support','n':int(v.size)})
 if np.any((v<0)|(v>1)):
  raise ValueError('histogram input must be windowed to [0,1]')
 bins=HISTEQ_BINS if nbins is None else nbins
 hist,edges=np.histogram(v,bins=bins,range=(0.0,1.0))
 cdf=np.cumsum(hist).astype(float)
 cdf/=cdf[-1]
 out[support]=np.interp(v,(edges[:-1]+edges[1:])/2,cdf)
 return(out,{'fallback':None,'n':int(v.size),'share_at_window_floor':float((v<=1e-06).mean()),'share_at_window_ceiling':float((v>=1-1e-06).mean())})

def window01(v, lo, hi):
    v = np.where(np.isfinite(v), v, lo); return ((np.clip(v, lo, hi) - lo) / (hi - lo)).astype(np.float32)
def _pnorm_channel(v, kind, levels, brain):
    valid = np.isfinite(v) & ((v > 0) if kind in ("cbf", "cbv", "mtt") else (v >= 0))
    rec = {"level": None, "n": 0, "median": None, "mad": None, "sd": None, "scale": None, "fallback": "window", "clip_low": None, "clip_high": None}
    ref = None
    for level, base in levels:
        r_ = base & valid
        if int(r_.sum()) >= PNORM_NN["min_ref_voxels"]: ref = r_; rec["level"] = level; break
    if ref is not None:
        vals = v[ref]; med = float(np.median(vals)); mad = float(np.median(np.abs(vals - med))); sd = float(np.std(vals)); ms = PNORM_NN["min_scale"][kind]
        scale, fb = (1.4826 * mad, "MAD") if np.isfinite(mad) and 1.4826 * mad >= ms else ((sd, "SD") if np.isfinite(sd) and sd >= ms else (None, "window"))
        rec.update({"n": int(ref.sum()), "median": med if np.isfinite(med) else None, "mad": mad, "sd": sd, "scale": scale, "fallback": fb})
        if scale is not None and np.isfinite(med):
            z = (np.where(np.isfinite(v), v, med) - med) / scale; zc = PNORM_NN["clip_z"]; zb = z[brain]
            rec.update({"clip_low": float(np.mean(zb < -zc)), "clip_high": float(np.mean(zb > zc))})
            return ((np.clip(z, -zc, zc) + zc) / (2 * zc)).astype(np.float32), rec
        rec["fallback"] = "window"
    return window01(v, *WINDOWS[kind]), rec


### Pipeline file: 0. checks before building or training  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
%%writefile -a /content/sprint13_pipeline.py
class PrerequisiteError(RuntimeError):
 pass

def canonical(obj):
 return(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=True,allow_nan=False)+'\n').encode('utf-8')

def sha_bytes(b):
 return hashlib.sha256(b).hexdigest()

def file_info(path):
 p=Path(path)
 h=hashlib.sha256()
 n=0
 with p.open('rb')as f:
  for b in iter(lambda:f.read(1048576),b''):
   h.update(b)
   n+=len(b)
 return{'sha256':h.hexdigest(),'bytes':n}

def need(test,message):
 if not test:
  raise PrerequisiteError(message)

def digest(value):
 return isinstance(value,str)and len(value)==64 and all((c in'0123456789abcdef'for c in value))

def rooted(root,relative):
 root=Path(root).resolve()
 r=Path(relative)
 need(not r.is_absolute()and r.parts and('..'not in r.parts),'E001')
 p=root/r
 need(p.resolve().is_relative_to(root),'E002')
 need(not any((x.is_symlink()for x in[p,*p.parents]if x!=root.parent)),'E003')
 return p

def read_verified(root,descriptor):
 need(set(descriptor)=={'path','sha256','bytes'},'E004')
 need(digest(descriptor['sha256'])and type(descriptor['bytes'])is int and(descriptor['bytes']>0),'E005')
 p=root.read(descriptor)if isinstance(root,DriveInputs)else rooted(root,descriptor['path'])
 need(file_info(p)=={k:descriptor[k]for k in('sha256','bytes')},'E006')
 return p

def write_once(root,rel,data):
 p=rooted(root,rel)
 p.parent.mkdir(parents=True,exist_ok=True)
 with p.open('xb')as f:
  f.write(data)
 return file_info(p)

def split_document(partitions,expected_count=99):
 need(isinstance(partitions,dict),'E007')
 keys=[k for k in partitions if k.startswith('101_')]
 need(set(keys)=={f'101_{k}'for k in range(5)},'E008')
 out=[]
 held=[]
 for k in range(5):
  row=partitions[f'101_{k}']
  tr=row['train']
  va=row['held']
  need(isinstance(tr,list)and isinstance(va,list)and tr and va,'E026')
  need(all((isinstance(x,str)and x and('/'not in x)and('\\'not in x)and(x not in('.','..'))for x in tr+va)),'E027')
  need(len(tr)==len(set(tr))and len(va)==len(set(va))and(not set(tr)&set(va)),'E028')
  out.append({'train':sorted(tr),'val':sorted(va)})
  held+=va
 need(len(held)==len(set(held))==expected_count,'E009')
 for row in out:
  need(set(row['train'])==set(held)-set(row['val']),'E029')
 return out

def _load_science():
 global np,pd,nib,resample_from_to,resample_to_output,gaussian_filter,uniform_filter,distance_transform_edt,cc_label,binary_fill_holes,binary_erosion,roc_auc_score,average_precision_score
 import numpy as np
 import pandas as pd
 import nibabel as nib
 from nibabel.processing import resample_from_to,resample_to_output
 from scipy.ndimage import gaussian_filter,uniform_filter,distance_transform_edt,label as cc_label,binary_fill_holes,binary_erosion
 from sklearn.metrics import roc_auc_score,average_precision_score

def verify_completed(output_root,completed):
 known={}
 for step,files in completed.items():
  need(isinstance(step,str)and isinstance(files,dict)and files,'E033')
  for rel,info in files.items():
   need(isinstance(info,dict)and set(info)=={'sha256','bytes'}and digest(info['sha256'])and(type(info['bytes'])is int)and(info['bytes']>0),'E039')
   need(rel not in known,'E040')
   need(file_info(rooted(output_root,rel))==info,'E041')
   known[rel]=info
 return known

def commit_step(root,contract,step,files):
 need(step not in contract.get('completed_steps',{}),'E020')
 old=verify_completed(root,contract.get('completed_steps',{}))
 need(files and(not set(files)&set(old)),'E021')
 records={r:file_info(rooted(root,r))for r in sorted(files)}
 contract['checkpoint'](step,records)
 contract.setdefault('completed_steps',{})[step]=records
 return records

def load_frozen_partitions(contract):
 f=contract['frozen_partitions']
 need(f['sha256']==PARTITIONS_SHA256,'E022')
 parts=metadata_file(contract,f)
 splits=split_document(parts)
 need(sha_bytes(canonical(splits))==SPLIT_SHA256,'E024')
 return splits

def transform_case(arrays,arm):
 need(arm==K_AR,'This delivery releases A1_repeat only')
 brain=arrays['brain']>0.5
 maps={k:arrays[k]for k in BASE6};outputs={};qc={}
 need(brain.any()and all(v.shape==brain.shape for v in maps.values()),'E025')
 for ch in BASE6:
  v=maps[ch];support,record=coverage_support(v,brain)
  x=window01(v,*WINDOWS[ch]);raw=v[brain&np.isfinite(v)]
  record.update(clip_low=float(np.mean(raw<=WINDOWS[ch][0]))if raw.size else None,clip_high=float(np.mean(raw>=WINDOWS[ch][1]))if raw.size else None)
  x=(x*brain).astype(np.float32)
  need(np.isfinite(x).all()and float(x[brain].std())>=1e-06,'E034')
  outputs[ch]=x;record['prepared_sd']=float(x[brain].std());qc[ch]=record
 return outputs,qc

def validate_full_folds(table,held):
 if set(held)!=set(range(5)):
  raise ValueError('expected frozen folds 0 through 4')
 expected=[(c,k,101)for k in range(5)for c in held[k]]
 if any((len(held[k])==0 for k in range(5)))or len(expected)!=99:
  raise ValueError('expected 99 members in five nonempty folds')
 members=[r[0]for r in expected]
 if len(set(members))!=99:
  raise ValueError('duplicate or overlapping frozen members')
 if not{'case','fold','shuffle'}.issubset(table.columns):
  raise ValueError('membership columns missing')
 keys=table[['case','fold','shuffle']]
 if len(keys)!=99 or keys.isna().any().any()or keys.case.duplicated().any():
  raise ValueError('missing, extra, null or duplicate rows')
 if set(keys.itertuples(index=False,name=None))!=set(expected):
  raise ValueError('row membership, fold or shuffle mismatch')

def verdict_from_intervals(base_ci,repeat_ci,complete=True):
 normalized=[]
 for ci in(base_ci,repeat_ci):
  try:
   if isinstance(ci,(str,bytes))or len(ci)!=3:
    raise ValueError('invalid interval')
   if any((hasattr(x,'__len__')and(not isinstance(x,(str,bytes)))for x in ci)):
    raise ValueError('invalid interval')
   arr=[float(x)for x in ci]
  except(ValueError,TypeError)as e:
   raise ValueError('invalid interval')from e
  if not all((math.isfinite(x)for x in arr))or not arr[1]<=arr[0]<=arr[2]:
   raise ValueError('malformed, reversed or nonfinite interval')
  normalized.append(arr)
 base_ci,repeat_ci=normalized
 if not complete:
  return'provisional'
 if base_ci[1]>0 and repeat_ci[1]>0:
  return'better'
 if base_ci[2]<0 and repeat_ci[2]<0:
  return'worse'
 return'no clear difference'

def source_paths(case,with_ctp=False):
 need(isinstance(case,str)and case and('/'not in case)and('\\'not in case)and(case not in('.','..')),'E043')
 d=f'train/derivatives/{case}'
 paths={k:f'{d}/ses-01/perfusion-maps/{case}_ses-01_space-ncct_{k}.nii.gz'for k in PERF_KINDS}
 paths.update(ncct=f'train/raw_data/{case}/ses-01/{case}_ses-01_ncct.nii.gz',cta=f'{d}/ses-01/{case}_ses-01_space-ncct_cta.nii.gz',lesion=f'{d}/ses-02/{case}_ses-02_space-ncct_lesion-msk.nii.gz',brain=f'brainmask/{case}_brainmask.nii.gz')
 if with_ctp:
  paths['ctp']=f'{d}/ses-01/{case}_ses-01_space-ncct_ctp.nii.gz'
 return paths

def source_inventory(input_root,cases,with_ctp,expected_sha256):
 need(isinstance(input_root,DriveInputs)and not with_ctp,'explicit in-place base inputs required')
 expected={c:dict(source_paths(c,False),cache=f'feature-cache-2mm-v2/{c}.npz')for c in cases}
 inventory=input_root.inventory
 need(sha_bytes(json.dumps(inventory,sort_keys=True,allow_nan=False).encode())==expected_sha256,'E045')
 need(set(inventory)=={rel for row in expected.values()for rel in row.values()}|{K_BPJ,K_BHJ},'E044')
 need(not input_root.verify()['missing'],'missing required file')
 return{'cases':{c:{k:dict(path=rel,**inventory[rel])for k,rel in row.items()}for c,row in expected.items()},'baseline_plans':dict(path=K_BPJ,**inventory[K_BPJ]),K_BH:dict(path=K_BHJ,**inventory[K_BHJ])}

def _input_manifest(input_root,contract,sel):
 splits=load_frozen_partitions(contract)
 cases=sorted((c for fold in splits for c in fold['val']))
 arms={f['arm']for f in contract[K_EP]['fits']if f[K_PI]==sel['id']}
 need(not arms&{K_AZ,K_AH},'U3 coverage held: no cohort-authenticated support definition')
 mapped=source_inventory(input_root,cases,False,sel[K_ICS])
 need(mapped['baseline_plans']['sha256']==BASE_PLANS_SHA256,'E046')
 need(mapped[K_BH]['sha256']==BASE_HANDOFF_SHA256,'E047')
 handoff=json.loads(read_verified(input_root,mapped[K_BH]).read_bytes())
 need(handoff['smoke']is False and sorted(handoff['cases'])==cases,'E048')
 need(handoff['arm']==K_ACMC and handoff['splits']==splits,'E049')
 need(handoff['source_inventory']=={'brain':'cad80ea68c35','cbf':'71905e6a3737','cbv':'678b1bfd5639','cta':'f3f200c85d00','lesion':'fbcfad347f52','mtt':'732e673174db','ncct':'b5137a3f1df6','tmax':'71fb01614363'},'E050')
 return mapped

def _native_paths(raw,pre,results):
 os.environ.update(nnUNet_raw=str(raw),nnUNet_preprocessed=str(pre),nnUNet_results=str(results),nnUNet_n_proc_DA=os.environ.get('COLAB_DA_WORKERS','2'))

def _load_case_sources(input_root,sources,arm):
 needed=set(BASE6)|{'brain','lesion','cache'}
 if arm in(K_AZ,K_AH):
  needed.add('ctp')
 need(set(sources)==needed,'E051')
 paths={k:read_verified(input_root,sources[k])for k in needed}
 ref=nib.load(str(paths['ncct']))
 arrays={}
 for k in needed-{'cache'}:
  img=nib.load(str(paths[k]))
  arr=np.asanyarray(img.dataobj)
  if k!='ctp'and arr.ndim==4 and(arr.shape[-1]==1):
   arr=arr[...,0]
  need(arr.shape[:3]==ref.shape[:3]and np.allclose(img.affine,ref.affine,atol=0.001,rtol=0),'E080')
  need(arr.ndim==(4 if k=='ctp'else 3),'E081')
  arrays[k]=arr
 brain=arrays['brain']>0.5
 ml=float(brain.sum()*np.prod(ref.header.get_zooms()[:3])/1000)
 need(400<=ml<=2500,'E052')
 need(np.isfinite(arrays['brain']).all()and np.isfinite(arrays['lesion']).all(),'E053')
 return(ref,arrays,paths)

def _dataset_name(arm):
 return f'Dataset{731+ARMS_ORDER.index(arm):03d}_ISLES24_{arm}'

def _prepared_paths(root,arm):
 ds=_dataset_name(arm)
 return(Path(root)/'raw'/ds,Path(root)/'preprocessed'/ds)

def _case_raw(input_root,output_root,contract,case,sources,arm):
 ref,arrays,paths=_load_case_sources(input_root,sources,arm)
 x,qc=transform_case(arrays,arm)
 cache=load_cache(str(paths['cache']),config_hash(FEATURE_CFG),case)
 truth=arrays['lesion']>0.5
 mapped=resample_from_to(nib.Nifti1Image(truth.astype(np.uint8),ref.affine),(cache['shape'],cache['affine']),order=0).get_fdata()>0.5
 need(dice(mapped,cache[K_LF])>=0.999,'E054')
 raw,pre=_prepared_paths(output_root,arm)
 written=[]
 for i,(ch,arr)in enumerate(x.items()):
  p=raw/'imagesTr'/f'{case}_{i:04d}.nii.gz'
  p.parent.mkdir(parents=True,exist_ok=True)
  need(not p.exists(),'E082')
  nib.save(nib.Nifti1Image(arr,ref.affine),str(p))
  written.append(p)
 p=raw/'labelsTr'/f'{case}.nii.gz'
 p.parent.mkdir(parents=True,exist_ok=True)
 need(not p.exists(),'E055')
 nib.save(nib.Nifti1Image(truth.astype(np.uint8),ref.affine),str(p))
 written.append(p)
 q=Path(output_root)/'qc'/f'{case}.json'
 write_once(output_root,q.relative_to(output_root).as_posix(),canonical({'channels':qc,'source_cache_dice':float(dice(mapped,cache[K_LF])),'brain_ml':float((arrays['brain']>0.5).sum()*np.prod(ref.header.get_zooms()[:3])/1000),'infarct_in_brain':float((truth&(arrays['brain']>0.5)).sum()/truth.sum())if truth.any()else 1.0}))
 written.append(q)
 return written

def _prepare_shared(input_root,output_root,contract,manifest,arm,splits,sel):
 from nnunetv2 .experiment_planning.dataset_fingerprint.fingerprint_extractor import DatasetFingerprintExtractor
 raw,pre=_prepared_paths(output_root,arm)
 ds=raw.name
 n=12 if arm==K_AM else 6
 dataset={'channel_names':{str(i):'noNorm'for i in range(n)},'labels':{'background':0,'infarct':1},'numTraining':99,'file_ending':'.nii.gz'}
 write_once(output_root,(raw/K_DJ).relative_to(output_root).as_posix(),canonical(dataset))
 DatasetFingerprintExtractor(ds,num_processes=2,verbose=False).run(overwrite_existing=False)
 bp=read_verified(input_root,manifest['baseline_plans'])
 bh=read_verified(input_root,manifest[K_BH])
 base=json.loads(bp.read_bytes())
 handoff=json.loads(bh.read_bytes())
 need(handoff['arm']==K_ACMC and handoff['splits']==splits,'E056')
 need(handoff['plans']==base,'E057')
 need(all((handoff['windows'][k]==list(WINDOWS[k])for k in BASE6)),'E058')
 need(handoff['smoke']is False and len(handoff['cases'])==99,'E059')
 if arm=='A1_L':
  from nnunetv2 .utilities.find_class_by_name import recursive_find_python_class
  import nnunetv2
  cls=recursive_find_python_class(str(Path(nnunetv2 .__file__).parent/'experiment_planning'),'nnUNetPlannerResEncL','nnunetv2.experiment_planning')
  need(cls is not None,'E083')
  planner=cls(ds)
  plans=planner.plan_experiment()
  need(plans[K_PN]==sel[K_PN],'E084')
 else:
  plans=copy.deepcopy(base)
  plans['dataset_name']=ds
  for cfg in plans[K_C].values():
   if K_NS in cfg:
    need(set(cfg[K_NS])=={K_N},'E092')
    cfg[K_NS]=[K_N]*n
   if'use_mask_for_norm'in cfg:
    cfg['use_mask_for_norm']=[False]*n
  fp=json.loads((pre/K_DFJ).read_bytes())
  if K_FIPPC in plans:
   plans[K_FIPPC]=fp[K_FIPPC]
  write_once(output_root,(pre/(sel[K_PN]+'.json')).relative_to(output_root).as_posix(),canonical(plans))
  for key in('patch_size','spacing','batch_size','architecture'):
   need(plans[K_C][K_3F][key]==base[K_C][K_3F][key],'E090')
 if not(pre/K_DJ).exists():
  write_once(output_root,(pre/K_DJ).relative_to(output_root).as_posix(),canonical(dataset))
 write_once(output_root,(pre/K_SFJ).relative_to(output_root).as_posix(),canonical(splits))
 write_once(output_root,'preparation.json',canonical({'arm':arm,'dataset':ds,K_PN:sel[K_PN],K_PS:file_info(pre/(sel[K_PN]+'.json'))['sha256'],K_PI:sel['id'],K_SS:sel[K_SS]}))
 return None

def _native_case(output_root,contract,case,arm,sel):
 from nnunetv2 .utilities.plans_handling.plans_handler import PlansManager
 raw,pre=_prepared_paths(output_root,arm)
 pm=PlansManager(str(pre/(sel[K_PN]+'.json')))
 cfg=pm.get_configuration(K_3F)
 worker=cfg.preprocessor_class(verbose=False)
 folder=pre/cfg.data_identifier
 folder.mkdir(parents=True,exist_ok=True)
 images=sorted((str(p)for p in(raw/'imagesTr').glob(case+'_*.nii.gz')))
 worker.run_case_save(str(folder/case),images,str(raw/'labelsTr'/f'{case}.nii.gz'),pm,cfg,json.loads((raw/K_DJ).read_bytes()))
 files=[p.relative_to(output_root).as_posix()for p in folder.glob(case+'.*')if p.is_file()]
 files+=[p.relative_to(output_root).as_posix()for p in folder.glob(case+'_seg.*')if p.is_file()]
 need(len(files)>=2,'E060')
 return files

def _arm(plan,sel):
 arms={f['arm']for f in plan['fits']if f[K_PI]==sel['id']}
 if arms=={K_AR,'A1_repeat2'}:
  return K_AR
 need(len(arms)==1,'E061')
 arm=next(iter(arms))
 need(arm not in(K_AZ,K_AH),'U3 coverage held')
 return arm

def _shared_publish(input_root,root,contract,manifest,arm,splits,sel):
 import shutil
 with tempfile.TemporaryDirectory(prefix='.staging-',dir=root)as td:
  stage=Path(td)
  _native_paths(stage/'raw',stage/'preprocessed',stage/'native-work')
  for case,sources in manifest['cases'].items():
   _case_raw(input_root,stage,contract,case,sources,arm)
  _prepare_shared(input_root,stage,contract,manifest,arm,splits,sel)
  _,src=_prepared_paths(stage,arm)
  _,dest=_prepared_paths(root,arm)
  dataset=json.loads((src/K_DJ).read_bytes())
  dataset['item4_fingerprint']=json.loads((src/K_DFJ).read_bytes())
  dataset['item4']={'arm':arm,K_PI:sel['id'],K_ICS:sel[K_ICS],K_SCS:sel[K_SCS],K_SS:sel[K_SS]}
  files=[]
  for name,raw in[(K_DJ,canonical(dataset)),(sel[K_PN]+'.json',(src/(sel[K_PN]+'.json')).read_bytes()),(K_SFJ,canonical(splits))]:
   rel=(dest/name).relative_to(root).as_posix()
   write_once(root,rel,raw)
   files.append(rel)
 commit_step(root,contract,'shared',files)

def _case_publish(input_root,root,contract,case,sources,arm,sel):
 import pickle
 with tempfile.TemporaryDirectory(prefix='.staging-',dir=root)as td:
  stage=Path(td)
  _native_paths(stage/'raw',stage/'preprocessed',stage/'native-work')
  _case_raw(input_root,stage,contract,case,sources,arm)
  raw,sp=_prepared_paths(stage,arm)
  _,pre=_prepared_paths(root,arm)
  for name in(K_DJ,sel[K_PN]+'.json'):
   write_once(stage,(sp/name).relative_to(stage).as_posix(),(pre/name).read_bytes())
  write_once(stage,(raw/K_DJ).relative_to(stage).as_posix(),(pre/K_DJ).read_bytes())
  emitted=_native_case(stage,contract,case,arm,sel)
  need(len(emitted)==3,'E085')
  prop=next((stage/r for r in emitted if r.endswith('.pkl')))
  with prop.open('rb')as f:
   properties=pickle.load(f)
  need('item4'not in properties,'E086')
  properties['item4']={'schema':'item4-private-scoring/v1','cache_bytes':read_verified(input_root,sources['cache']).read_bytes(),'cache_info':{k:sources['cache'][k]for k in('sha256','bytes')},'truth_nifti':(raw/'labelsTr'/f'{case}.nii.gz').read_bytes(),'qc':json.loads((stage/'qc'/f'{case}.json').read_bytes())}
  with prop.open('wb')as f:
   pickle.dump(properties,f,protocol=4)
  files=[]
  for rel in emitted:
   write_once(root,rel,(stage/rel).read_bytes())
   files.append(rel)
 commit_step(root,contract,'native-'+case,files)

def _preprocess(input_root,output_root,contract):
 plan,rt,sel=bound_plan(contract,True)
 _load_science()
 arm=_arm(plan,sel)
 manifest=_input_manifest(input_root,contract,sel)
 splits=load_frozen_partitions(contract)
 cases=sorted((c for s in splits for c in s['val']))
 need(set(manifest['cases'])==set(cases),'E062')
 root=Path(output_root).resolve()
 root.mkdir(parents=True,exist_ok=True)
 completed=contract.setdefault('completed_steps',{})
 known=verify_completed(root,completed)
 recover_uncommitted(root,known,contract['attempt_root'])
 need(set(completed)<={'shared'}|{'native-'+c for c in cases},'E064')
 need(not completed or'shared'in completed,'E065')
 if'shared'not in completed:
  _shared_publish(input_root,root,contract,manifest,arm,splits,sel)
 for case in cases:
  if'native-'+case not in completed:
   _case_publish(input_root,root,contract,case,manifest['cases'][case],arm,sel)
 return validate_preprocessing(root,contract)

def _private_scoring(properties):
 payload=properties.get('item4',{})
 need(payload.get('schema')=='item4-private-scoring/v1','E066')
 b=payload['cache_bytes']
 info=payload['cache_info']
 need(isinstance(b,bytes)and info=={'sha256':sha_bytes(b),'bytes':len(b)},'E067')
 qc=payload['qc']
 need(qc['source_cache_dice']>=0.999 and 400<=qc['brain_ml']<=2500,'E068')
 need(all((math.isfinite(x['prepared_sd'])and x['prepared_sd']>=1e-06 for x in qc['channels'].values())),'E069')
 need(isinstance(payload['truth_nifti'],bytes)and payload['truth_nifti'],'E070')
 return payload

def _validate_preprocessing(output_root,contract):
 plan,rt,sel=bound_plan(contract,True)
 _load_science()
 arm=_arm(plan,sel)
 splits=load_frozen_partitions(contract)
 cases=sorted((c for s in splits for c in s['val']))
 files=verify_completed(output_root,contract.get('completed_steps',{}))
 root=Path(output_root).resolve()
 need(set(contract['completed_steps'])=={'shared'}|{'native-'+c for c in cases},'E071')
 need(len(files)==300 and set(files)=={p.relative_to(root).as_posix()for p in root.rglob('*')if p.is_file()},'E072')
 _,pre=_prepared_paths(root,arm)
 need((pre/K_SFJ).read_bytes()==canonical(splits),'E073')
 metadata=json.loads((pre/K_DJ).read_bytes())['item4']
 need(metadata=={'arm':arm,K_PI:sel['id'],K_ICS:sel[K_ICS],K_SCS:sel[K_SCS],K_SS:sel[K_SS]},'E074')
 from nnunetv2 .utilities.plans_handling.plans_handler import PlansManager
 from nnunetv2 .training.dataloading.nnunet_dataset import infer_dataset_class
 pm=PlansManager(str(pre/(sel[K_PN]+'.json')))
 cfg=pm.get_configuration(K_3F)
 need(list(cfg.normalization_schemes)==[K_N]*(12 if arm==K_AM else 6),'E075')
 folder=pre/cfg.data_identifier
 dataset=infer_dataset_class(str(folder))(str(folder))
 need(len(dataset.identifiers)==len(cases)and set(dataset.identifiers)==set(cases),'E076')
 for case in cases:
  loaded=dataset.load_case(case)
  data=np.asarray(loaded[0][:])
  seg=np.asarray(loaded[1][:])
  props=loaded[-1]
  need(data.ndim==4 and data.shape[0]==(12 if arm==K_AM else 6),'E087')
  need(seg.shape[1:]==data.shape[1:]and np.isfinite(data).all()and np.isfinite(seg).all(),'E088')
  need(np.isin(seg,[-1,0,1]).all()and'class_locations'in props,'E089')
  _private_scoring(props)
 meta={K_PS:file_info(pre/(sel[K_PN]+'.json'))['sha256']}
 return preprocessing_receipts(root,contract,sel,meta,cases,pre,folder,files)

def preprocessing_receipts(root,contract,sel,meta,cases,pre,folder,committed):
 rt=contract['runtime']
 root=Path(root)
 case_files={}
 for case in cases:
  members={}
  for role,bases in{'image':[case+'.b2nd',case+'.npy',case+'.npz'],'segmentation':[case+'_seg.b2nd',case+'_seg.npy',case+'_seg.npz'],'properties':[case+'.pkl']}.items():
   matches=[folder/b for b in bases if(folder/b).is_file()]
   need(len(matches)==1,'E091')
   q=matches[0]
   members[role]=dict(path=q.relative_to(root).as_posix(),**file_info(q))
  case_files[case]=members
 shared={k:dict(path=q.relative_to(root).as_posix(),**file_info(q))for k,q in{'dataset':pre/K_DJ,'plans':pre/(sel[K_PN]+'.json'),'splits':pre/K_SFJ}.items()}
 files={d['path']:{k:d[k]for k in('sha256','bytes')}for group in[*case_files.values(),shared]for d in group.values()}
 need(files==committed,'E077')
 module_sha=rt['execution'][K_MS]
 need(module_sha==sha_bytes(Path(__file__).read_bytes()),'E078')
 need(shared['plans']['sha256']==meta[K_PS]and shared['splits']['sha256']==sel[K_SS],'E079')
 identities={K_CS:sel[K_CS],K_ICS:sel[K_ICS],K_ES:contract[K_P][K_ES],K_PCS:module_sha}
 proposed=dict(identities,schema='modal-item4-preprocessed/v1',status='VALIDATED',source_capture_sha256=sel[K_SCS],plans_sha256=meta[K_PS],case_files=case_files,shared_files=shared)
 validation=dict(identities,schema='modal-preprocessing-validation/v1',status='PASS',errors=[],run_id=rt['run_id'],spec_sha256=rt['spec_sha256'],validator_sha256=module_sha,files=files)
 return{'proposed':proposed,K_V:validation}

def timing_record(fit_id,records,comparability_id):
 need(records,'E093')
 epochs=[r['epoch']for r in records]
 need(len(epochs)==len(set(epochs))and all((type(e)is int and e>=0 for e in epochs)),'E094')
 need(epochs==list(range(epochs[0],epochs[0]+len(epochs))),'E095')
 need(all((type(r.get(K_TI))is int and r[K_TI]==250 and(type(r.get(K_VI))is int)and(r[K_VI]==50)for r in records)),'E096')
 need(all((math.isfinite(r['elapsed'])and r['elapsed']>0 and math.isfinite(r[K_LW])and(0<=r[K_LW]<=r['elapsed'])for r in records)),'E097')
 wait=sum((r[K_LW]for r in records))
 elapsed=sum((r['elapsed']for r in records))
 return{'schema':'experiment-epoch-timing/v1','fit_id':fit_id,'completed_epochs':len(records),'elapsed_training_seconds':elapsed,'real_epoch':True,K_LNB:wait/elapsed<=0.1,'comparability_id':comparability_id,'measured_loader_wait_seconds':wait,'epoch_records':records}

def _trainer_class():
 from nnunetv2 .training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
 class TimedIterator:
  def __init__(s,inner,owner):
   s.inner=inner
   s.owner=owner
  def __iter__(s):
   return s
  def __next__(s):
   t=time.perf_counter()
   value=next(s.inner)
   s.owner._loader_wait+=time.perf_counter()-t
   return value
  def __getattr__(s,k):
   return getattr(s.inner,k)
 class ReviewedTrainer(nnUNetTrainer):
  def __init__(s,plans,configuration,fold,dataset_json,device):
   super().__init__(plans,configuration,fold,dataset_json,device)
   s._measurements=[]
   s._loader_wait=0.0
   s._loaded_epoch=None
  def get_dataloaders(s):
   a,b=super().get_dataloaders()
   return(TimedIterator(a,s),TimedIterator(b,s))
  def train_step(s,batch):
   s._lease.guard()
   result=super().train_step(batch)
   need(np.isfinite(result['loss']).all(),'Nonfinite training loss')
   s._train_batches+=1
   return result
  def validation_step(s,batch):
   s._lease.guard()
   result=super().validation_step(batch)
   need(np.isfinite(result['loss']).all(),'Nonfinite validation loss')
   s._val_batches+=1
   return result
  def load_checkpoint(s,filename_or_checkpoint):
   super().load_checkpoint(filename_or_checkpoint)
   need(s.current_epoch>=0 and s.optimizer is not None,'E136')
   need(s.current_epoch==0 or s.optimizer.state_dict()['state'],'E137')
   s._loaded_epoch=int(s.current_epoch)
  def on_epoch_start(s):
   import torch
   s._lease.guard()
   _synchronize_device(s.device)
   s._epoch_t=time.perf_counter()
   s._loader_wait=0.0
   s._measured_epoch=int(s.current_epoch)
   s._train_batches=0
   s._val_batches=0
   super().on_epoch_start()
  def on_epoch_end(s):
   import torch
   _synchronize_device(s.device)
   core=time.perf_counter()-s._epoch_t
   super().on_epoch_end()
   _synchronize_device(s.device)
   elapsed=time.perf_counter()-s._epoch_t
   need(s._train_batches==250 and s._val_batches==50,'E138')
   rec={'epoch':s._measured_epoch,'elapsed':elapsed,'core_synchronized_seconds':core,'native_seconds':float(s.logger.get_value('epoch_end_timestamps',step=-1)-s.logger.get_value('epoch_start_timestamps',step=-1)),K_LW:s._loader_wait,K_TI:s._train_batches,K_VI:s._val_batches}
   s.print_to_log_file('ITEM4_EPOCH '+canonical(rec).decode().strip(),add_timestamp=False)
   s._measurements.append(rec)
   if s.current_epoch % 5 == 0 or s.current_epoch == 1:
    s._epoch_checkpoint()
 return ReviewedTrainer

def _score_native(input_root,tr,arm,fold,splits,private_root):
 raw,pre=_prepared_paths(input_root,K_AR if arm=='A1_repeat2'else arm)
 plans=tr.plans_manager.plans
 need(plans['image_reader_writer']=='SimpleITKIO','E099')
 records=[]
 validation=Path(tr.output_folder)/K_V
 for case in splits[fold]['val']:
  segimg=nib.load(str(validation/f'{case}.nii.gz'))
  seg=np.asanyarray(segimg.dataobj)>0.5
  with np.load(validation/f'{case}.npz',allow_pickle=False)as z:
   p=np.transpose(z['probabilities'][1].astype(np.float32),(2,1,0))
  need(p.shape==seg.shape and np.isfinite(p).all()and((p>=0)&(p<=1)).all(),'E124')
  need(dice(p>0.5,seg)>=0.99,'E125')
  cc=_fit_cache(pre,tr,case,private_root)
  pred=resample_from_to(nib.Nifti1Image(seg.astype(np.uint8),segimg.affine),(cc['shape'],cc['affine']),order=0).get_fdata()>0.5
  prob=resample_from_to(nib.Nifti1Image(p,segimg.affine),(cc['shape'],cc['affine']),order=1).get_fdata().astype(np.float32)
  row=patient_scores(cc,prob[cc['mask']],pred)
  fp=int((pred&~cc[K_LF]&cc['mask']).sum())
  tn=int((~pred&~cc[K_LF]&cc['mask']).sum())
  row['specificity']=tn/(tn+fp)if tn+fp else None
  row.update(case=case,fold=fold,shuffle=101)
  need(all((math.isfinite(row[k])for k in('dice','lesion_f1','abs_vol_err_ml','signed_vol_err_ml'))),'E126')
  records.append({k:None if isinstance(v,(float,np.floating))and(not math.isfinite(v))else v.item()if isinstance(v,np.generic)else v for k,v in row.items()})
 need(len(records)==len(splits[fold]['val']),'E100')
 Path(private_root,'heldout-scores.json').write_bytes(canonical(records))
 metrics={}
 for k in('dice','lesion_f1','abs_vol_err_ml','signed_vol_err_ml','precision','recall','specificity','asd_mm','fm_auroc','fm_avg_precision','lesion_ml_outside_mask'):
  vals=[float(r[k])for r in records if r.get(k)is not None and math.isfinite(r[k])]
  metrics[k]={'n_defined':len(vals),'mean':sum(vals)/len(vals)if vals else None}
 return{'n_heldout':len(records),'fold':fold,'metrics':metrics,'exploratory':True,'uncertainty':'Single-fold development result; patient resampling does not estimate training variance.'}

def _fit_payload(pre,tr,case):
 import pickle
 folder=pre/tr.configuration_manager.data_identifier
 with(folder/(case+'.pkl')).open('rb')as f:
  return _private_scoring(pickle.load(f))

def _fit_cache(pre,tr,case,private_root):
 payload=_fit_payload(pre,tr,case)
 p=Path(private_root)/'scoring'/f'{case}.npz'
 if p.exists():
  need(file_info(p)==payload['cache_info'],'E127')
 else:
  write_once(private_root,p.relative_to(private_root).as_posix(),payload['cache_bytes'])
 return load_cache(str(p),config_hash(FEATURE_CFG),case)

def _fit_truth(pre,tr,cases,work):
 base=Path(work)/'native-inputs'
 base.mkdir(exist_ok=True)
 raw=(pre/K_DJ).read_bytes()
 meta=json.loads(raw)
 fingerprint=meta['item4_fingerprint']
 need(isinstance(fingerprint,dict)and fingerprint,'E101')
 for name,b in[(K_DJ,raw),(K_SFJ,(pre/K_SFJ).read_bytes()),(K_DFJ,canonical(fingerprint))]:
  p=base/name
  if p.exists():
   need(p.read_bytes()==b,'E139')
  else:
   write_once(base,name,b)
 tr.preprocessed_dataset_folder_base=str(base)
 folder=base/'gt_segmentations'
 folder.mkdir(exist_ok=True)
 for case in cases:
  b=_fit_payload(pre,tr,case)['truth_nifti']
  p=folder/f'{case}.nii.gz'
  if p.exists():
   need(p.read_bytes()==b,'E140')
  else:
   write_once(base,p.relative_to(base).as_posix(),b)
 tr.gt_segmentations_folder=str(folder)

def cpu_capacity():
 count=os.cpu_count()
 affinity=len(os.sched_getaffinity(0))
 quota=None
 source=None
 p=Path('/sys/fs/cgroup/cpu.max')
 if p.is_file():
  q,period=p.read_text().split()
  quota=None if q=='max'else int(q)/int(period)
  source='cgroup-v2'
 else:
  p=Path('/sys/fs/cgroup/cpu/cpu.cfs_quota_us')
  q=Path('/sys/fs/cgroup/cpu/cpu.cfs_period_us')
  if p.is_file()and q.is_file():
   a=int(p.read_text())
   quota=None if a<0 else a/int(q.read_text())
   source='cgroup-v1'
 need(source is not None and count is not None and(affinity>0),'E146')
 return{'logical_cpu_count':count,'affinity_logical_cpus':affinity,'quota_cpu_equivalents':quota,'quota_source':source,'effective_logical_capacity':min(affinity,quota)if quota is not None else affinity}

def _synchronize_device(device):
 import torch
 if device.type=='cuda':
  torch.cuda.synchronize(device)
 else:
  need(device.type=='cpu','E163')

def membership_guard(contract):
 splits=load_frozen_partitions(contract)
 members={c for row in splits for c in row['val']}
 need(contract['colab']['development_manifest']['sha256']==COHORT_SHA256,'development manifest binding')
 dev=metadata_file(contract,contract['colab']['development_manifest'])
 excluded=metadata_file(contract,contract['colab']['exclusions'])
 need(len(dev['cases'])==len(set(dev['cases']))==99 and set(dev['cases'])==members,'positive membership failed')
 need(len(excluded['cases'])==len(set(excluded['cases']))==49 and not members&set(excluded['cases']),'exclusion membership failed')
 return splits

def file_hash(path):return file_info(path)["sha256"]

def verify_native_source():
 from importlib.metadata import version
 import inspect
 from nnunetv2 .training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
 if version('nnunetv2')!='2.8.1'or file_hash(inspect.getfile(nnUNetTrainer))!=TRAINER_SOURCE_SHA256:
  raise ValueError('NNUNET_NATIVE_SOURCE_CHANGED')

def strict_json(raw):
 def pairs(items):
  out={}
  for key,value in items:
   need(key not in out,'duplicate JSON key');out[key]=value
  return out
 def invalid(value):raise PrerequisiteError('nonfinite JSON')
 def number(value):
  v=float(value);need(math.isfinite(v),'nonfinite JSON');return v
 return json.loads(raw,object_pairs_hook=pairs,parse_constant=invalid,parse_float=number)

def checked_path(value,missing=False):
 import stat
 p=Path(value)
 need(p.is_absolute()and str(p)==str(value)and'..'not in p.parts,'noncanonical absolute path')
 for q in reversed([p,*p.parents]):
  try:s=q.lstat()
  except FileNotFoundError:
   need(missing,'missing path');continue
  need(not stat.S_ISLNK(s.st_mode),'symlink refused')
  need(stat.S_ISDIR(s.st_mode)or stat.S_ISREG(s.st_mode),'special path refused')
  if stat.S_ISREG(s.st_mode):need(q==p and s.st_nlink==1,'file alias refused')
 return p

def metadata_file(contract,descriptor):
 need(set(descriptor)=={'path','sha256','bytes'},'metadata descriptor')
 p=checked_path(descriptor['path'])
 root=checked_path(contract['colab']['metadata_root'])
 need(p.is_relative_to(root)and p!=root,'metadata outside authorized root')
 need(digest(descriptor['sha256'])and type(descriptor['bytes'])is int and descriptor['bytes']>0,'metadata identity')
 raw=p.read_bytes()
 need({'sha256':sha_bytes(raw),'bytes':len(raw)}=={k:descriptor[k]for k in('sha256','bytes')},'metadata mismatch')
 return strict_json(raw)

class DriveInputs:
 def __init__(self,mapping,inventory,allowed):
  need(set(mapping)=={'schema','roots','files'}and mapping['schema']=='colab-in-place-inputs/v1','mapping schema')
  need(isinstance(mapping['files'],list)and mapping['files'],'mapping rows')
  need(mapping['roots']==allowed and allowed,'authorized roots changed')
  self.roots={k:checked_path(v)for k,v in allowed.items()}
  roots=list(self.roots.values())
  need(len(set(roots))==len(roots)and all(not a.is_relative_to(b)for i,a in enumerate(roots)for j,b in enumerate(roots)if i!=j),'ambiguous roots')
  self.rows={};self.inventory=inventory;physical=set()
  for row in mapping['files']:
   need(set(row)=={'logical_path','root','relative_path','sha256','bytes'},'mapping row schema')
   logical=row['logical_path'];relative=row['relative_path']
   need(logical in inventory and logical not in self.rows,'unexpected/duplicate logical path')
   need(row['root']in self.roots,'unauthorized root')
   need(isinstance(relative,str)and relative and not relative.startswith('/')and all(x not in('','.','..')for x in relative.split('/'))and'\\'not in relative,'noncanonical relative path')
   need(type(row['bytes'])is int and{k:row[k]for k in('sha256','bytes')}==inventory[logical],'mapping differs from frozen identity')
   p=self.roots[row['root']]/relative
   need(p not in physical,'duplicate physical mapping');physical.add(p)
   self.rows[logical]=row
  need(set(self.rows)==set(inventory),'incomplete mapping; not a missing-file observation')
 def path(self,logical):
  row=self.rows[logical];root=self.roots[row['root']]
  p=checked_path(str(root/row['relative_path']),missing=True)
  need(p.is_relative_to(root)and p!=root,'input outside authorized root')
  return p
 def verify(self):
  missing=[];observed={};inodes=set()
  for logical in sorted(self.rows):
   p=self.path(logical)
   try:s=p.stat()
   except FileNotFoundError:
    missing.append(dict(self.rows[logical]));continue
   import stat
   need(stat.S_ISREG(s.st_mode)and s.st_nlink==1,'input is not a unique regular file')
   inode=(s.st_dev,s.st_ino);need(inode not in inodes,'physical alias');inodes.add(inode)
   actual=file_info(p)
   need(actual==self.inventory[logical],'frozen input mismatch: '+logical)
   observed[logical]=actual
  return{'schema':'colab-input-check/v1','arm':'A1_repeat','status':'MISSING'if missing else'VERIFIED','missing':missing,'observed':observed,'ctp_required':False,'discovery_performed':False}
 def read(self,descriptor):
  logical=descriptor['path'];need(logical in self.rows,'unmapped read')
  need({k:descriptor[k]for k in('sha256','bytes')}==self.inventory[logical],'unfrozen read')
  p=self.path(logical);need(file_info(p)==self.inventory[logical],'input changed at read')
  return p

def in_place_inputs(contract):
 splits=membership_guard(contract)
 cases=sorted(c for row in splits for c in row['val'])
 cfg=contract['colab']
 inventory=metadata_file(contract,cfg['frozen_inventory'])
 need(sha_bytes(json.dumps(inventory,sort_keys=True,allow_nan=False).encode())==BASE_INPUT_SHA256,'frozen inventory mismatch')
 required={p for c in cases for p in dict(source_paths(c,False),cache=f'feature-cache-2mm-v2/{c}.npz').values()}|{K_BPJ,K_BHJ}
 need(set(inventory)==required and len(required)==893,'released-arm inventory mismatch')
 for item in inventory.values():
  need(set(item)=={'sha256','bytes'}and digest(item['sha256'])and type(item['bytes'])is int and item['bytes']>0,'frozen file identity')
 mapping=metadata_file(contract,cfg['input_mapping'])
 excluded=metadata_file(contract,cfg['exclusions'])['cases']
 need(all(not any(x in str(mapping['roots'][row['root']])+'/'+row['relative_path']for x in excluded)for row in mapping['files']),'excluded identifier in physical mapping')
 return DriveInputs(mapping,inventory,cfg['authorized_input_roots'])

def actual_environment():
 import sys,platform,subprocess,importlib.metadata as md,torch
 packages={d.metadata['Name']:d.version for d in md.distributions()if d.metadata['Name']}
 ram=os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')
 capacity=cpu_capacity()
 need(capacity['effective_logical_capacity']>=2 and ram>=16*1024**3,'insufficient observed CPU/RAM')
 need(torch.cuda.is_available()and'A100'in torch.cuda.get_device_name(0),'A100 required')
 gpu=torch.cuda.get_device_properties(0)
 need(gpu.total_memory>=39*1024**3,'A100 memory')
 driver=subprocess.run(['nvidia-smi','--id=0','--query-gpu=driver_version,name,memory.total','--format=csv,noheader'],capture_output=True,text=True,check=True,timeout=10).stdout.strip()
 return{'schema':'colab-environment-observation/v1','unix':time.time(),'python':sys.version,'platform':platform.platform(),'packages':packages,'torch_cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_bytes':gpu.total_memory,'driver':driver,'cpu':capacity,'ram_bytes':ram,'augmentation_workers_requested':2,'module_sha256':file_info(__file__)['sha256']}

EPOCHS = 250
SEED = 101
WORKERS = 2
PLANS_NAME = 'nnUNetResEncUNetMPlans'
METADATA_ROOT = '/content/drive/MyDrive/isles-pilot/colab-1ad5fafc220b3d65f576ce08/package'
RESULTS_ROOT = '/content/drive/MyDrive/isles-pilot/colab-1ad5fafc220b3d65f576ce08/results-full250'
INPUT_REFERENCE_PATH = METADATA_ROOT + '/accepted-input-reference.json'
INPUT_REFERENCE_SHA256 = "8ff800b4ecc9d6f3cdc6e5e41144e84acfcb7fe93d0e5d0138d4560ee6e16d04"
ADDED_PACKAGES = {'nnunetv2':'2.8.1','numpy':'2.2.6','pandas':'2.2.3','scipy':'1.15.3',
                  'nibabel':'5.4.2','scikit-learn':'1.6.1','SimpleITK':'2.5.2'}

def delivery_check(arm,folds):
 need(arm == K_AR and isinstance(folds,tuple) and folds and
      all(type(f) is int and f in range(5) for f in folds) and len(set(folds))==len(folds),
      'Invalid built-in arm/folds')
 return folds

def install_dependencies():
 import sys,subprocess,importlib.metadata as md
 need(sys.version_info[:2] == (3,13),'This delivery targets default Colab Python 3.13')
 existing={k:md.version(k) for k in ('torch','torchvision','torchaudio') if importlib.util.find_spec(k)}
 need('torch' in existing,'Colab preinstalled torch required')
 constraints=Path('/content/preinstalled-torch-constraints.txt')
 constraints.write_text(''.join(k+'=='+v+'\n' for k,v in existing.items()))
 subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-c',str(constraints),
                 *[k+'=='+v for k,v in ADDED_PACKAGES.items()]],check=True)
 need(all(md.version(k)==v for k,v in existing.items()),'Preinstalled torch stack changed')


def compatible_environment(env):
 import sys,importlib.metadata as md
 return {'python':list(sys.version_info[:2]),
         'packages':{k:md.version(k) for k in (*ADDED_PACKAGES,'torch','dynamic-network-architectures',
                       'batchgenerators','batchgeneratorsv2','blosc2')},
         'cuda':env['torch_cuda'],'cudnn':env['cudnn'],'gpu_family':'A100',
         'native_trainer_sha256':TRAINER_SOURCE_SHA256}


def load_input_reference():
 need(digest(INPUT_REFERENCE_SHA256),'Invalid accepted input reference binding')
 p=checked_path(INPUT_REFERENCE_PATH)
 raw=p.read_bytes()
 need(sha_bytes(raw)==INPUT_REFERENCE_SHA256,'Accepted input reference changed')
 ref=strict_json(raw)
 need(set(ref)=={'schema','frozen_partitions','colab','fit_lock_ids'} and
      ref['schema']=='colab-accepted-input-reference/v1','Input reference schema')
 need(ref['colab']['metadata_root']==METADATA_ROOT,'Metadata root changed')
 need(set(ref['colab'])=={'metadata_root','development_manifest','exclusions','frozen_inventory','input_mapping','authorized_input_roots','preprocessing_references'},'Only accepted metadata descriptors may be imported')
 need(set(ref['fit_lock_ids'])=={str(k) for k in range(5)} and
      len(set(ref['fit_lock_ids'].values()))==5,'Five distinct server lock IDs required')
 membership_guard(ref)
 return ref


def scientific_config(fold):
 return {'arm':K_AR,'fold':fold,'seed':SEED,'epochs':EPOCHS,'lr_horizon':250,
         'training_iterations':250,'validation_iterations':50,'workers':WORKERS,
         'fingerprint_workers':2,'checkpoint_every_epochs':5,'windows':{k:list(WINDOWS[k]) for k in BASE6},
         'configuration':K_3F,'plans_name':PLANS_NAME,'normalization':[K_N]*6,
         'baseline_plans_sha256':BASE_PLANS_SHA256,'input_inventory_sha256':BASE_INPUT_SHA256,
         'cohort_sha256':COHORT_SHA256,'split_sha256':SPLIT_SHA256,
         'source_capture_sha256':'b252b308815e983df2d9d86975a6d590e20d2600ea2f118696263797e8509f5c'}


def make_contract(ref,fold,env,attempt):
 cfg=scientific_config(fold)
 sel={'id':'prep-A1_repeat',K_PN:PLANS_NAME,K_ICS:BASE_INPUT_SHA256,K_CS:COHORT_SHA256,
      K_SS:SPLIT_SHA256,K_SCS:cfg['source_capture_sha256']}
 c={'frozen_partitions':ref['frozen_partitions'],'colab':ref['colab'],
    K_EP:{'fits':[{'arm':K_AR,'fold':fold,K_PI:sel['id']}],K_P:[sel]},
    K_P:{K_ES:sha_bytes(canonical(env))},'attempt_root':str(attempt),
    'runtime':{'run_id':RUN_ID,'spec_sha256':sha_bytes(canonical(cfg)),
               'execution':{K_MS:file_info(__file__)['sha256']}}}
 return c


def bound_plan(contract,preprocessing=False):
 plan=contract[K_EP]; rt=contract['runtime']; sel=plan[K_P][0]
 need(sel[K_ICS]==BASE_INPUT_SHA256 and sel[K_CS]==COHORT_SHA256 and sel[K_SS]==SPLIT_SHA256,
      'Scientific input binding changed')
 need(rt['execution'][K_MS]==file_info(__file__)['sha256'],'Module identity changed')
 return plan,rt,sel


def recover_uncommitted(root,known,attempt):
 import shutil
 unknown=[p for p in root.rglob('*') if p.is_file() and p.relative_to(root).as_posix() not in known]
 if unknown:
  archive=Path(attempt)/'uncommitted'/uuid.uuid4().hex
  for p in unknown:
   dest=archive/p.relative_to(root);dest.parent.mkdir(parents=True,exist_ok=True);shutil.move(str(p),dest)


def preprocess(input_root,output_root,contract):
 membership_guard(contract)
 need(str(input_root)==METADATA_ROOT,'Read existing metadata in place')
 return _preprocess(in_place_inputs(contract),output_root,contract)


def validate_preprocessing(output_root,contract):
 membership_guard(contract)
 return _validate_preprocessing(output_root,contract)


class DriveCAS:
 """Conditional server update of a prebound private lock object."""
 def __init__(self,file_id):
  import google.auth
  from google.auth.transport.requests import AuthorizedSession
  import re
  need(isinstance(file_id,str) and re.fullmatch(r'[A-Za-z0-9_-]+',file_id),'Invalid lock file ID')
  credentials,_=google.auth.default()
  self.session=AuthorizedSession(credentials)
  self.url='https://www.googleapis.com/drive/v2/files/'+file_id
 def read(self):
  from email.utils import parsedate_to_datetime
  r=self.session.get(self.url,timeout=30);r.raise_for_status();obj=r.json()
  need(obj.get('etag') and r.headers.get('Date'),'Drive server version/time missing')
  need(obj.get('labels',{}).get('trashed') is False and obj.get('mimeType')!='application/vnd.google-apps.shortcut','Invalid lock object')
  return obj['etag'],parsedate_to_datetime(r.headers['Date']).timestamp(),strict_json(obj.get('description') or '{}')
 def replace(self,etag,value):
  r=self.session.patch(self.url,headers={'If-Match':etag},json={'description':canonical(value).decode()},timeout=30)
  if r.status_code==412:raise PrerequisiteError('Same-fit writer collision or lease lost')
  r.raise_for_status()


class FitLease:
 """Server-time lease; immutable attempt paths fence out stale writers."""
 def __init__(self,backend,binding):
  import threading
  self.backend=backend;self.binding=binding;self.owner=uuid.uuid4().hex
  self.mutex=threading.RLock();self.stop=threading.Event();self.error=None
  self.thread=None;self.state=None;self.deadline=0
 def acquire(self):
  with self.mutex:
   etag,now,state=self.backend.read()
   need(not state or state.get('binding')==self.binding,'Same-fit code/input/config/environment mismatch')
   need(not state.get('owner') or state.get('expires',0)<=now,'This fit is running in another session')
   if not state:state={'binding':self.binding,'generation':0,'checkpoints':{}}
   state=dict(state,owner=self.owner,expires=now+300,generation=state['generation']+1)
   self.backend.replace(etag,state);self.state=state;self.deadline=time.monotonic()+240
   self.guard()
  return self
 def _update(self,changes=None,release=False):
  with self.mutex:
   etag,now,state=self.backend.read()
   need(state.get('binding')==self.binding and state.get('owner')==self.owner and
        state.get('generation')==self.state['generation'] and state.get('expires',0)>now,'Lease expired or ownership changed')
   state.update(changes or {});state['expires']=now+300
   if release:state.update(owner=None,expires=0)
   self.backend.replace(etag,state);self.state=state;self.deadline=time.monotonic()+240
 def guard(self):
  if self.error:raise PrerequisiteError('Lease heartbeat failed; rerun after recovery') from self.error
  need(self.state is not None and self.state.get('owner')==self.owner,'Writer lease required')
  need(time.monotonic()<self.deadline,'Lease renewal deadline passed')
 def publish(self,key,value):
  self.guard();self._update({key:value})
 def heartbeat(self):
  import threading
  def loop():
   while not self.stop.wait(45):
    try:self._update()
    except BaseException as e:self.error=e;return
  self.thread=threading.Thread(target=loop,daemon=True);self.thread.start()
 def close(self):
  self.stop.set()
  if self.thread:self.thread.join(timeout=35)
  if self.error is None:
   try:self._update(release=True)
   except Exception:pass


def immutable_record(root,path,obj):
 data=canonical(obj);write_once(root,path,data)
 return dict(path=path,**file_info(Path(root)/path))


def read_record(root,ref):
 p=read_verified(root,ref)
 return strict_json(p.read_bytes())


class Checkpoints:
 def __init__(self,root,attempt,lease):
  self.root=Path(root);self.attempt=Path(attempt);self.lease=lease
 def publish(self,key,write,metadata):
  self.lease.guard();need(key in ('latest','best','final'),'Checkpoint key')
  p=self.attempt/'checkpoints'/(uuid.uuid4().hex+'.pth');p.parent.mkdir(exist_ok=True)
  start=time.perf_counter();write(p)
  with p.open('rb') as f:os.fsync(f.fileno())
  info=file_info(p)
  rec=dict(info,path=p.relative_to(self.root).as_posix(),metadata=metadata,binding=self.lease.binding,
           owner=self.lease.owner,generation=self.lease.state['generation'],write_seconds=time.perf_counter()-start)
  side=immutable_record(self.root,rec['path']+'.json',rec)
  need(file_info(p)==info,'Checkpoint readback failed')
  with self.lease.mutex:
   self.lease._update()
   checkpoints=dict(self.lease.state['checkpoints']);checkpoints[key]=side
   self.lease.publish('checkpoints',checkpoints)
  return rec
 def select(self,key):
  self.lease.guard()
  record=read_record(self.root,self.lease.state['checkpoints'][key])
  need(record['binding']==self.lease.binding,'Checkpoint binding mismatch')
  p=read_verified(self.root,{k:record[k] for k in ('path','sha256','bytes')})
  need(record['metadata']['total_epochs']==EPOCHS and 0<=record['metadata']['next_epoch']<=EPOCHS,'Checkpoint epoch mismatch')
  if key=='final':need(record['metadata']['next_epoch']==EPOCHS,'Incomplete final checkpoint')
  return p,record


def resume_choice(checkpoints):
 return 'final' if 'final' in checkpoints else 'latest' if 'latest' in checkpoints else None


def run_fit(trainer,progress):
 verify_native_source();progress.lease.guard()
 need(trainer.num_epochs==EPOCHS and not trainer.is_ddp and not trainer.disable_checkpointing,'Full250 single GPU required')
 trainer.save_every=5
 original_save=trainer.save_checkpoint
 def publish(key,next_epoch):
  import random,torch
  state_epoch=trainer.current_epoch
  need(type(next_epoch) is int and 0<=next_epoch<=EPOCHS,'Invalid next epoch')
  def write(path):
   trainer.current_epoch=next_epoch-1
   try:original_save(str(path))
   finally:trainer.current_epoch=state_epoch
   blob=torch.load(str(path),map_location='cpu',weights_only=False)
   need(blob['current_epoch']==next_epoch,'Native checkpoint epoch convention changed')
   blob['delivery_state']={'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
                           'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                           'measurements':copy.deepcopy(trainer._measurements)}
   torch.save(blob,str(path))
  return progress.publish(key,write,{'next_epoch':next_epoch,'total_epochs':EPOCHS,'native_version':'2.8.1'})
 def native_save(self,filename):
  name=Path(filename).name
  if name=='checkpoint_latest.pth':return None
  kinds={'checkpoint_best.pth':'best','checkpoint_final.pth':'final'}
  need(name in kinds,'Unexpected native checkpoint')
  return publish(kinds[name],self.current_epoch+1)
 trainer.save_checkpoint=types.MethodType(native_save,trainer)
 trainer._epoch_checkpoint=lambda:publish('latest',trainer.current_epoch)
 trainer._lease=progress.lease
 choice=resume_choice(progress.lease.state['checkpoints'])
 try:
  if choice:
   import torch,random
   path,rec=progress.select(choice)
   blob=torch.load(str(path),map_location='cpu',weights_only=False)
   trainer.load_checkpoint(blob)
   need(trainer.current_epoch==rec['metadata']['next_epoch'],'Loaded optimizer epoch mismatch')
   saved=blob['delivery_state'];trainer._measurements=saved['measurements']
   need([x['epoch'] for x in trainer._measurements]==list(range(trainer.current_epoch)),'Checkpoint timing history mismatch')
   random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng']);torch.set_rng_state(saved['torch_rng']);torch.cuda.set_rng_state_all(saved['cuda_rng'])
  else:
   trainer.initialize();publish('latest',0)
  resumed=int(trainer.current_epoch)
  if choice!='final':trainer.run_training()
  _,final=progress.select('final')
  need(trainer.current_epoch==EPOCHS and len(trainer._measurements)==EPOCHS,'Full training incomplete')
  progress.lease.guard()
  trainer.perform_actual_validation(save_probabilities=True)
  return {'resumed_from_epoch':resumed,'final_checkpoint':final,'bitwise_replay_claimed':False}
 finally:trainer.save_checkpoint=original_save


def completed_fit(root,lease):
 ref=lease.state.get('completed')
 if ref is None:return None
 summary=read_record(root,ref)
 need(summary['binding']==lease.binding and summary['epochs']==EPOCHS and summary['status']=='COMPLETE','Completed fit binding changed')
 verify_completed(root,{'complete':summary['files']})
 return summary


def execute_fold(ref,fold,env,compat):
 import random,torch
 root=Path(RESULTS_ROOT)/K_AR/f'fold{fold}'
 checked_path(str(root),missing=True)
 for r in [METADATA_ROOT,*ref['colab']['authorized_input_roots'].values()]:
  r=checked_path(r);need(not root.is_relative_to(r) and not r.is_relative_to(root),'Output/input overlap')
 binding={'module_sha256':file_info(__file__)['sha256'],'input_reference_sha256':INPUT_REFERENCE_SHA256,
          'scientific_config':scientific_config(fold),'compatible_environment':compat}
 lease=FitLease(DriveCAS(ref['fit_lock_ids'][str(fold)]),binding).acquire();lease.heartbeat()
 root.mkdir(parents=True,exist_ok=True)
 attempt=root/'attempts'/lease.owner;attempt.mkdir(parents=True,exist_ok=False)
 started=time.perf_counter();status='FAILED'
 try:
  immutable_record(root,(attempt/'start.json').relative_to(root).as_posix(),
                   {'binding':binding,'environment':env,'started_unix':time.time(),'generation':lease.state['generation']})
  write_once(attempt,'execution.py',Path(__file__).read_bytes())
  write_once(attempt,'executed-notebook.ipynb',Path('/content/delivery-executed.ipynb').read_bytes())
  c=make_contract(ref,fold,compat,attempt)
  input_start=time.perf_counter()
  inputs=in_place_inputs(c);verification=inputs.verify()
  write_once(attempt,'input-verification.json',canonical(dict(verification,elapsed_seconds=time.perf_counter()-input_start,
               development_count=99,excluded_count=49,membership_checked=True,reference_sha256=INPUT_REFERENCE_SHA256)))
  need(not verification['missing'],'Required mapped input missing; see private input-verification.json')
  for j,desc in enumerate(ref['colab']['preprocessing_references']):
   metadata_file(c,desc);write_once(attempt,f'historical-preprocessing-{j}.json',Path(desc['path']).read_bytes())
  done=completed_fit(root,lease)
  if done is not None:
   Checkpoints(root,attempt,lease).select('final')
   status='VERIFIED_SKIP';return done
  prep_start=time.perf_counter()
  if lease.state.get('prepared'):
   receipt=read_record(root,lease.state['prepared'])
   need(receipt['binding']==binding,'Preprocessing binding mismatch')
   prep=rooted(root,receipt['path']);c['completed_steps']=receipt['steps']
   validation=validate_preprocessing(prep,c)
  else:
   prep=attempt/'prepared';c['completed_steps']={}
   def save_step(step,files):
    lease.guard();write_once(attempt,'steps/'+step+'.json',canonical(files))
   c['checkpoint']=save_step
   validation=preprocess(METADATA_ROOT,prep,c)
   receipt={'binding':binding,'path':prep.relative_to(root).as_posix(),'steps':c['completed_steps'],'validation':validation}
   manifest=immutable_record(root,(attempt/'preprocessing.json').relative_to(root).as_posix(),receipt)
   lease.publish('prepared',manifest)
  prep_seconds=time.perf_counter()-prep_start
  lease.guard()
  random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
  os.environ['COLAB_DA_WORKERS']=str(WORKERS)
  raw,pre=_prepared_paths(prep,K_AR);work=attempt/'work';work.mkdir()
  _native_paths(Path(prep)/'raw',Path(prep)/'preprocessed',work/'native-results')
  plans=strict_json((pre/(PLANS_NAME+'.json')).read_bytes())
  tr=_trainer_class()(plans,K_3F,fold,strict_json((pre/K_DJ).read_bytes()),device=torch.device('cuda',0))
  Path(tr.output_folder).mkdir(parents=True,exist_ok=True)
  tr.log_file=str(work/'training.log');tr.num_epochs=EPOCHS
  need(tr.num_iterations_per_epoch==250 and tr.num_val_iterations_per_epoch==50,'Iteration counts changed')
  splits=membership_guard(c);_fit_truth(pre,tr,splits[fold]['val'],work)
  progress=Checkpoints(root,attempt,lease);native_start=time.perf_counter()
  provenance=run_fit(tr,progress)
  losses=[tr.logger.get_value('train_losses',step=i) for i in range(EPOCHS)]
  need(len(losses)==EPOCHS and all(math.isfinite(float(x)) for x in losses),'Nonfinite/incomplete training losses')
  timing=timing_record(f'A1_repeat-f{fold}',tr._measurements,'full99-250epochs-native-v1')
  timing.update(preprocessing_seconds=prep_seconds,native_training_and_validation_seconds=time.perf_counter()-native_start)
  score_start=time.perf_counter();metrics=_score_native(prep,tr,K_AR,fold,splits,work)
  timing['scoring_seconds']=time.perf_counter()-score_start
  write_once(attempt,'metrics.json',canonical(metrics));write_once(attempt,'timing.json',canonical(timing))
  write_once(attempt,'checkpoint-provenance.json',canonical(provenance))
  files={p.relative_to(root).as_posix():file_info(p) for p in attempt.rglob('*') if p.is_file()}
  files[lease.state['prepared']['path']]={k:lease.state['prepared'][k] for k in ('sha256','bytes')}
  final=provenance['final_checkpoint'];files[final['path']]={k:final[k] for k in ('sha256','bytes')}
  summary={'schema':'colab-full250-summary/v1','run_id':RUN_ID,'arm':K_AR,'fold':fold,'epochs':EPOCHS,
           'status':'COMPLETE','binding':binding,'files':files,'metrics':metrics,'wall_seconds':time.perf_counter()-started,
           'public_export':False,'scientific_acceptance':False,'checkpoint_provenance':provenance,
           'limitations':'Exposed development cohort. One retraining realization; not a training variance estimate. Resume is not bitwise worker RNG replay.'}
  summary_ref=immutable_record(root,(attempt/'run-summary.json').relative_to(root).as_posix(),summary)
  write_once(attempt,'run-summary.sha256',(summary_ref['sha256']+'\n').encode())
  lease.publish('completed',summary_ref);status='COMPLETE'
  return summary
 finally:
  try:
   import sys,traceback
   error=traceback.format_exc() if sys.exc_info()[0] else None
   write_once(attempt,'attempt-end.json',canonical({'status':status,'wall_seconds':time.perf_counter()-started,'ended_unix':time.time(),'error':error}))
  finally:lease.close()


def main(input_root,output_root,contract):
 need(str(input_root)==METADATA_ROOT and str(output_root)==RESULTS_ROOT,'Built-in roots changed')
 need(isinstance(contract,dict) and set(contract)=={'arm','folds'},'Delivery contract fields')
 delivery_check(contract['arm'],contract['folds'])
 return run_delivery(contract['arm'],contract['folds'])

def run_delivery(arm,folds):
 delivery_check(arm,folds)
 os.umask(0o077)
 os.environ['nnUNet_def_n_proc']='2'
 _load_science();verify_native_source()
 need(_fx()=={'both_empty':(1.,0),'truth_only':(0.,1),'diagonal_contact_one_piece':(1.,0),'truth_one_pred_two_split':(2/3,1),'one_matched_one_fp':(2/3,1),'iou_0.2_boundary':(1.,0)},'Metric invariants failed')
 env=actual_environment();compat=compatible_environment(env);ref=load_input_reference()
 import torch
 torch.set_num_threads(2)
 results=[]
 for fold in folds:
  results.append(execute_fold(ref,fold,env,compat))
 if set(folds)==set(range(5)):
  rows=[]
  for r in results:
   foldroot=Path(RESULTS_ROOT)/K_AR/f"fold{r['fold']}"
   matches=[p for p in r['files'] if p.endswith('/heldout-scores.json')]
   need(len(matches)==1,'Heldout score record missing')
   rows.extend(strict_json(rooted(foldroot,matches[0]).read_bytes()))
  validate_full_folds(pd.DataFrame(rows),{k:s['val'] for k,s in enumerate(membership_guard(ref))})
 print('Selected folds complete and verified. Results remain private on Drive.')
 return results

def synthetic_tests():
 """Controller-only synthetic doubles; no patient/GPU/network evidence."""
 from unittest.mock import patch
 class MemoryCAS:
  def __init__(self):self.etag=0;self.now=1000.;self.value={}
  def read(self):return str(self.etag),self.now,copy.deepcopy(self.value)
  def replace(self,etag,value):
   need(etag==str(self.etag),'CAS conflict')
   self.value=copy.deepcopy(value);self.etag+=1
 class DeliveryTests(unittest.TestCase):
  def test_main_interface(self):
   c={'arm':K_AR,'folds':(2,)}
   with patch.dict(globals(),{'run_delivery':lambda a,f:(a,f)}):
    self.assertEqual(main(METADATA_ROOT,RESULTS_ROOT,c),(K_AR,(2,)))
    for i,o,v in [('other',RESULTS_ROOT,c),(METADATA_ROOT,'other',c),(METADATA_ROOT,RESULTS_ROOT,{})]:
     with self.assertRaises(PrerequisiteError):main(i,o,v)
  def test_fixed_delivery_and_folds(self):
   self.assertEqual(EPOCHS,250);self.assertEqual(SEED,101)
   for fs in ((0,1,2,3,4),*( (k,) for k in range(5))):self.assertEqual(delivery_check(K_AR,fs),fs)
   for arm,fs in [('other',(0,)),(K_AR,()),(K_AR,(0,0)),(K_AR,(True,)),(K_AR,(5,))]:
    with self.assertRaises(PrerequisiteError):delivery_check(arm,fs)
   roots=[str(Path(RESULTS_ROOT)/K_AR/f'fold{k}') for k in range(5)]
   self.assertEqual(len(set(roots)),5)
   for k in range(5):
    c=scientific_config(k);self.assertEqual((c['epochs'],c['lr_horizon'],c['fold']),(250,250,k))
  def test_real_main_consumes_fold_selection(self):
   calls=[]
   global np
   import numpy as np
   import sys,types
   with patch.dict(sys.modules,{'torch':types.SimpleNamespace(set_num_threads=lambda n:None)}), patch.dict(globals(),{'_load_science':lambda:None,'verify_native_source':lambda:None,
        'actual_environment':lambda:{},'compatible_environment':lambda e:{},'load_input_reference':lambda:{},
        'execute_fold':lambda r,f,e,c:calls.append(f) or {}}):
    main(METADATA_ROOT,RESULTS_ROOT,{'arm':K_AR,'folds':(1,3)})
   self.assertEqual(calls,[1,3])
  def test_membership_and_hash_refusal(self):
   with tempfile.TemporaryDirectory() as d:
    root=Path(d);members=[f'SYNTHETIC_{i:03d}' for i in range(99)]
    parts={f'101_{k}':{'train':[x for i,x in enumerate(members) if i%5!=k],
                      'held':[x for i,x in enumerate(members) if i%5==k]} for k in range(5)}
    def desc(name,value):
     p=root/name;p.write_bytes(canonical(value));return dict(path=str(p),**file_info(p))
    split=desc('partitions.json',parts);dev=desc('dev.json',{'cases':members})
    excluded=desc('excluded.json',{'cases':[f'SYNTHETIC_X{i:03d}' for i in range(49)]})
    c={'frozen_partitions':split,'colab':{'metadata_root':str(root),'development_manifest':dev,'exclusions':excluded}}
    with patch.dict(globals(),{'PARTITIONS_SHA256':split['sha256'],'COHORT_SHA256':dev['sha256'],
                              'SPLIT_SHA256':sha_bytes(canonical(split_document(parts)))}):
     self.assertEqual(len(membership_guard(c)),5)
     (root/'dev.json').write_bytes(b'changed')
     with self.assertRaises(PrerequisiteError):membership_guard(c)
     c['colab']['development_manifest']=desc('dev.json',{'cases':members})
     c['colab']['exclusions']=desc('excluded.json',{'cases':[members[0]]+[f'SYNTHETIC_X{i:03d}' for i in range(48)]})
     with self.assertRaises(PrerequisiteError):membership_guard(c)
  def test_mapping_no_discovery_or_aliases(self):
   with tempfile.TemporaryDirectory() as d:
    root=Path(d);(root/'one').write_bytes(b'synthetic');info=file_info(root/'one')
    row=dict(logical_path='one',root='source',relative_path='one',**info)
    mapping=dict(schema='colab-in-place-inputs/v1',roots={'source':str(root)},files=[row])
    inp=DriveInputs(mapping,{'one':info},mapping['roots'])
    self.assertFalse(inp.verify()['missing'])
    for change in ({'relative_path':'../one'},{'sha256':'0'*64}):
     with self.assertRaises(PrerequisiteError):DriveInputs(dict(mapping,files=[dict(row,**change)]),{'one':info},mapping['roots'])
    (root/'one').write_bytes(b'corrupt')
    with self.assertRaises(PrerequisiteError):inp.verify()
    (root/'one').unlink();self.assertEqual(len(inp.verify()['missing']),1)
    (root/'target').write_bytes(b'synthetic');(root/'one').symlink_to(root/'target')
    with self.assertRaises(PrerequisiteError):inp.verify()
  def test_lease_collision_expiry_and_fencing(self):
   backend=MemoryCAS();binding={'test':'synthetic'}
   a=FitLease(backend,binding).acquire()
   with self.assertRaises(PrerequisiteError):FitLease(backend,binding).acquire()
   backend.now+=301;b=FitLease(backend,binding).acquire()
   with self.assertRaises(PrerequisiteError):a.publish('completed',{})
   b.publish('proof','new owner');b.close()
   with self.assertRaises(PrerequisiteError):FitLease(backend,{'different':'code'}).acquire()
  def test_checkpoint_resume_corruption_mismatch_completed_skip(self):
   with tempfile.TemporaryDirectory() as d:
    root=Path(d);attempt=root/'attempt';attempt.mkdir()
    backend=MemoryCAS();lease=FitLease(backend,{'test':'synthetic'}).acquire()
    p=Checkpoints(root,attempt,lease)
    self.assertIsNone(resume_choice(lease.state['checkpoints']))
    rec=p.publish('latest',lambda x:x.write_bytes(b'optimizer'),{'next_epoch':5,'total_epochs':250})
    self.assertEqual(resume_choice(lease.state['checkpoints']),'latest');self.assertEqual(p.select('latest')[1],rec)
    (root/rec['path']).write_bytes(b'corrupt')
    with self.assertRaises(PrerequisiteError):p.select('latest')
    rec=p.publish('final',lambda x:x.write_bytes(b'final'),{'next_epoch':250,'total_epochs':250})
    self.assertEqual(resume_choice(lease.state['checkpoints']),'final')
    write_once(root,'scores.json',b'[]\n')
    record={'binding':lease.binding,'epochs':250,'status':'COMPLETE','files':{'scores.json':file_info(root/'scores.json')}}
    ref=immutable_record(root,'done.json',record);lease.publish('completed',ref)
    self.assertEqual(completed_fit(root,lease),record)
    (root/'scores.json').write_bytes(b'corrupt')
    with self.assertRaises(PrerequisiteError):completed_fit(root,lease)
    lease.close()
  def test_no_meter_or_segment_gate(self):
   import inspect,ast
   text='\n'.join(inspect.getsource(f) for f in (main,run_delivery,execute_fold,run_fit,preprocess,make_contract))
   nodes=list(ast.walk(ast.parse(text)))
   self.assertFalse(any(isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id=='input' for n in nodes))
   for forbidden in ('accounting_guard','cu_start','cu_end','SmokeInterruption','_COLAB_DEADLINE','segment_reserved_micro_usd'):
    self.assertNotIn(forbidden,text)
   self.assertEqual(scientific_config(0)['epochs'],250)
  def test_preserved_metric_fixtures(self):
   global np
   import numpy as np
   values=_fx()
   expected={'both_empty':(1.,0),'truth_only':(0.,1),'diagonal_contact_one_piece':(1.,0),
             'truth_one_pred_two_split':(2/3,1),'one_matched_one_fp':(2/3,1),'iou_0.2_boundary':(1.,0)}
   self.assertEqual(set(values),set(expected))
   for k,v in expected.items():self.assertAlmostEqual(values[k][0],v[0]);self.assertEqual(values[k][1],v[1])
 return unittest.TestSuite([unittest.defaultTestLoader.loadTestsFromTestCase(DeliveryTests)])

if __name__ == '__main__':
 import sys
 need(len(sys.argv)==3,'Internal delivery invocation required')
 main(METADATA_ROOT,RESULTS_ROOT,{'arm':sys.argv[1],'folds':tuple(int(x) for x in sys.argv[2].split(','))})


### Pipeline file: 1-2. dataset, plans and preprocessing  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
# Code consolidated in the preceding importable pipeline. No execution here.


### Pipeline file: 3. training  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
# Code consolidated in the preceding importable pipeline. No execution here.


### Pipeline file: 4. scoring  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
# Code consolidated in the preceding importable pipeline. No execution here.


### Pipeline file: 5. results and stage receipt  
*This cell only writes code to `/content/sprint13_pipeline.py`; the driver below runs it once per stage.*

In [ ]:
# Code consolidated in the preceding importable pipeline. No execution here.


## Run the selected folds and save private results


In [ ]:
import importlib.util, subprocess, sys, json
from pathlib import Path
print(f"{DELIVERY_ARM}: folds {DELIVERY_FOLDS}; 250 epochs each")
n_folds = len(DELIVERY_FOLDS)
print(f"Historical training estimate: {2.69*n_folds:.2f}–{2.74*n_folds:.2f} hours; "
      f"{2.69*6.77*n_folds:.1f}–{2.74*6.77*n_folds:.1f} CU at reported 6.77 CU/hour, PLUS overhead. Not a guarantee.")
from google.colab import drive, auth, _message
drive.mount('/content/drive')
auth.authenticate_user()
spec = importlib.util.spec_from_file_location('pipeline', '/content/sprint13_pipeline.py')
pipeline = importlib.util.module_from_spec(spec)
sys.modules['pipeline'] = pipeline
spec.loader.exec_module(pipeline)
pipeline.delivery_check(DELIVERY_ARM, DELIVERY_FOLDS)
pipeline.load_input_reference()
pipeline.install_dependencies()
# Capture actual cell sources privately; exclude transient outputs and credentials.
nb = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        cell['outputs'] = []
        cell['execution_count'] = None
Path('/content/delivery-executed.ipynb').write_text(json.dumps(nb, ensure_ascii=False))
subprocess.run([sys.executable, '/content/sprint13_pipeline.py', DELIVERY_ARM,
                ','.join(str(f) for f in DELIVERY_FOLDS)], check=True)


## Combined results (Sprint 13 arms against the Sprint 12 baseline)


In [ ]:
import math
def validate_full_folds(table, held):
 if set(held) != set(range(5)):
  raise ValueError('expected frozen folds 0 through 4')
 expected = [(c, k, 101) for k in range(5) for c in held[k]]
 if any((len(held[k]) == 0 for k in range(5))) or len(expected) != 99:
  raise ValueError('expected 99 members in five nonempty folds')
 members = [r[0] for r in expected]
 if len(set(members)) != 99:
  raise ValueError('duplicate or overlapping frozen members')
 if not {'case', 'fold', 'shuffle'}.issubset(table.columns):
  raise ValueError('membership columns missing')
 keys = table[['case', 'fold', 'shuffle']]
 if len(keys) != 99 or keys.isna().any().any() or keys.case.duplicated().any():
  raise ValueError('missing, extra, null or duplicate rows')
 if set(keys.itertuples(index=False, name=None)) != set(expected):
  raise ValueError('row membership, fold or shuffle mismatch')
def verdict_from_intervals(base_ci, repeat_ci, complete=True):
 normalized = []
 for ci in (base_ci, repeat_ci):
  try:
   if isinstance(ci, (str, bytes)) or len(ci) != 3:
    raise ValueError('invalid interval')
   if any((hasattr(x, '__len__') and (not isinstance(x, (str, bytes))) for x in ci)):
    raise ValueError('invalid interval')
   arr = [float(x) for x in ci]
  except (ValueError, TypeError) as e:
   raise ValueError('invalid interval') from e
  if not all((math.isfinite(x) for x in arr)) or not arr[1] <= arr[0] <= arr[2]:
   raise ValueError('malformed, reversed or nonfinite interval')
  normalized.append(arr)
 base_ci, repeat_ci = normalized
 if not complete:
  return 'provisional'
 if base_ci[1] > 0 and repeat_ci[1] > 0:
  return 'better'
 if base_ci[2] < 0 and repeat_ci[2] < 0:
  return 'worse'
 return 'no clear difference'

# The system analyses private fold returns against the frozen baseline as they arrive.
